# Model Training with Leakage


In [ ]:
from pathlib import Path
import numpy as np
import h5py

import random

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC

from sklearn.metrics import roc_auc_score, average_precision_score


In [ ]:

records = {}

# =====================================================
# LOAD SEIZURE FILES
# =====================================================

seizure_files = sorted(
    Path("Features/Seizure").glob("chb01_*_seizure.mat")
)

for file in seizure_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]
        y = f["all_labels"][:].squeeze()

    if X.shape[0] == 1530:
        X = X.T

    record_name = file.stem.replace("_seizure", "")

    records[record_name] = (X, y)

    print(
        "[SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

# =====================================================
# LOAD NON-SEIZURE FILES
# =====================================================

non_files = sorted(
    Path("Features/Non_Seizure").glob("chb01_*_features.mat")
)

for file in non_files:

    with h5py.File(file, "r") as f:
        X = f["all_feats"][:]

    if X.shape[0] == 1530:
        X = X.T

    y = np.zeros(X.shape[0], dtype=int)

    record_name = file.stem.replace("_features", "")

    records[record_name] = (X, y)

    print(
        "[NON-SEIZURE]",
        record_name,
        X.shape,
        y.shape,
        "positives =", int(np.sum(y))
    )

print("\nTotal records:", len(records))

In [ ]:
# =====================================================
# SPLIT SEIZURE / NON-SEIZURE RECORDS
# =====================================================
seizure_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) > 0
}

non_records = {
    k:v for k,v in records.items()
    if np.sum(v[1]) == 0
}


print("Seizure:", len(seizure_records))
print("Non seizure:", len(non_records))

In [ ]:
# =====================================================
# SELECT NON-SEIZURE RECORDS FOR TRAINING
# =====================================================

r = 0
random.seed(42)


selected_non = random.sample(
    list(non_records.keys()),
    r
)


print("Selected non seizure:")
print(selected_non)

In [ ]:
def create_train_test(test_record):

    X_train = []
    y_train = []
    train_groups = []


    # -------------------------
    # seizure records
    # -------------------------

    for name, (X,y) in seizure_records.items():

        if name == test_record:
            continue


        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # -------------------------
    # non seizure records
    # -------------------------

    for name in selected_non:

        X,y = non_records[name]

        X_train.append(X)
        y_train.append(y)

        train_groups.extend(
            [name] * len(y)
        )



    # combine

    X_train = np.vstack(X_train)
    y_train = np.concatenate(y_train)

    train_groups = np.array(train_groups)



    # test

    X_test, y_test = seizure_records[test_record]


    return (
        X_train,
        y_train,
        X_test,
        y_test,
        train_groups
    )

## 1. Test loading files

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, average_precision_score, recall_score,
    precision_score, f1_score, confusion_matrix
)

# =====================================================
# CONFIG — the ONLY things you change between runs
# =====================================================
SEED = 42
DATA_POOL = seizure_records   # <-- swap to `records` for the all-files condition
TUNE = True                   # <-- True = grid search; False = use FIXED_PARAMS
FIXED_PARAMS = {"k": 100, "C": 10, "gamma": "scale"}

param_grid = {
    "k": [10, 30, 50, 100],
    "C": [0.1, 1, 10],
    "gamma": ["scale", 0.01],
}

# =====================================================
# BUILD DATA  (identical for every run on the same pool)
# =====================================================
X_all = np.vstack([X for X, y in DATA_POOL.values()])
y_all = np.hstack([y for X, y in DATA_POOL.values()]).astype(int)
print("Total windows:", X_all.shape)
print("Class counts :", np.bincount(y_all))

# =====================================================
# SPLIT — fixed 70/20/10, fixed seed, identical every run
# =====================================================
# 1. carve off test FIRST — 10% of everything, set aside
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X_all, y_all, test_size=0.10, random_state=SEED, stratify=y_all
)

# 2. split the remaining 90% into train and val
#    val = 0.20/0.90 ≈ 0.2222 of trainval  -> 20% of the whole
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=2/9, random_state=SEED, stratify=y_trainval
)
print("Train/Val/Test:", X_train.shape[0], X_val.shape[0], X_test.shape[0])
print("Test positives:", int(y_test.sum()))


def build(params):
    return Pipeline([
        ("select", SelectKBest(f_classif, k=params["k"])),
        ("scale",  StandardScaler()),
        ("svm",    SVC(kernel="rbf", C=params["C"], gamma=params["gamma"],
                       class_weight="balanced")),   # no probability=True
    ])


# =====================================================
# SELECT HYPERPARAMETERS (on validation only)
# =====================================================
if TUNE:
    best_score, best_params = -np.inf, None
    for params in ParameterGrid(param_grid):
        m = build(params).fit(X_train, y_train)
        s = m.decision_function(X_val)
        ap = average_precision_score(y_val, s)
        if ap > best_score:
            best_score, best_params = ap, params
    print("\nBEST PARAMS:", best_params, "| val AUPRC:", round(best_score, 4))
else:
    best_params = FIXED_PARAMS
    print("\nFIXED PARAMS:", best_params)

# =====================================================
# FINAL MODEL: refit on train+val, score test ONCE
# =====================================================
X_fit = np.vstack([X_train, X_val])
y_fit = np.concatenate([y_train, y_val])
final = build(best_params).fit(X_fit, y_fit)

scores = final.decision_function(X_test)        # one consistent score function
y_pred = (scores >= 0).astype(int)              # decision_function threshold = 0

auc   = roc_auc_score(y_test, scores)
auprc = average_precision_score(y_test, scores)
sens  = recall_score(y_test, y_pred)
prec  = precision_score(y_test, y_pred, zero_division=0)
f1    = f1_score(y_test, y_pred, zero_division=0)
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print("\n===== RESULT =====")
print(f"ROC-AUC     : {auc:.3f}")
print(f"AUPRC       : {auprc:.3f}")
print(f"Sensitivity : {sens:.3f}")
print(f"Precision   : {prec:.3f}")
print(f"F1          : {f1:.3f}")
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")
print(f"Baseline AUPRC (prevalence): {y_test.mean():.4f}")